# Análise estatística

In [1]:
import itertools
import pandas as pd
import scikit_posthocs as sp
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare, mannwhitneyu

In [2]:
df = pd.read_csv('analysis_mean_min_max_std.csv', sep=",")

In [3]:
def is_smaller_instance(filename):
    # Extrai o número da instância do nome do arquivo
    num = int(filename.split('/')[-1].split('-')[0].replace('FIS', ''))
    return num <= 14

In [4]:
df_bigger = df[~df['filename'].apply(is_smaller_instance)]

In [5]:
df_smaller = df[df['filename'].apply(is_smaller_instance)]

In [6]:
# Preparar os dados
def friedman_test(df):
    pivot_table = df.pivot(index='filename', columns='algorithm', values='mean')

    unique_algorithms = df['algorithm'].nunique()
    
    stat, p = friedmanchisquare(*pivot_table.values.T)
    print(f"Friedman Test: Chi-square = {stat}, p-value = {p}")
    
    if p < 0.05:
        nemenyi_results = sp.posthoc_nemenyi_friedman(df.pivot(index='filename', columns='algorithm', values='mean'))
        display(nemenyi_results)
    else:
        print("O teste de Friedman não foi significativo, portanto, o teste post-hoc não foi realizado.")

In [7]:
friedman_test(df_smaller)

Friedman Test: Chi-square = 31.641630901287567, p-value = 6.22735516717976e-07


,CheapestInsertion,GeneticImproved,Memetic,NearestNeighborhood
CheapestInsertion,1.000000,0.001000,0.001000,0.004195
GeneticImproved,0.001000,1.000000,0.654713,0.282368
Memetic,0.001000,0.654713,1.000000,0.900000
NearestNeighborhood,0.004195,0.282368,0.900000,1.000000


In [8]:
friedman_test(df_bigger)

Friedman Test: Chi-square = 30.920000000000016, p-value = 8.836642957681827e-07


,CheapestInsertion,GeneticImproved,Memetic,NearestNeighborhood
CheapestInsertion,1.000000,0.255307,0.010066,0.405611
GeneticImproved,0.255307,1.000000,0.001000,0.900000
Memetic,0.010066,0.001000,1.000000,0.001000
NearestNeighborhood,0.405611,0.900000,0.001000,1.000000


## Mann Whitney

In [9]:
df = pd.read_csv('2023-08-20.csv', sep=";")

In [10]:
df_bigger = df[~df['filename'].apply(is_smaller_instance)]
df_smaller = df[df['filename'].apply(is_smaller_instance)]

In [11]:
algorithms = ['Memetic', 'GeneticImproved']

In [12]:
instances = list(df['filename'].unique())

In [13]:
results = []

for instance in instances:
    if is_smaller_instance(instance):
        df_instances = df_smaller[df_smaller['filename'] == instance]
    else:
        df_instances = df_bigger[df_bigger['filename'] == instance]
        
    df_memetic = df_instances[df_instances['algorithm'] == algorithms[0]]
    df_genetic = df_instances[df_instances['algorithm'] == algorithms[1]]
    stat, p = mannwhitneyu(df_memetic['cost'], df_genetic['cost'])
    results.append([instance, stat, p])

results_df = pd.DataFrame(results, columns=['Instance', 'U-statistic', 'P-value'])

In [14]:
results_df.to_csv('mannwhitneyu-memetic-vs-genetic.csv')